# Explaining the model

This notebook shows where the delivered model looks when it makes a decision. It places a Grad-CAM activation map on top of each B-scan, so that a reader can see whether the model relies on the retinal pathology or on an artefact of the acquiring device.

This notebook is a template. The activation map is computed by the functions in `ocular.explain`, three of which are left as stubs to be implemented. The cells marked TODO run once those functions are filled in. The visualisation helper `overlay` is already implemented.

## Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch

from ocular import config, data, explain
from ocular import model as omodel
from ocular.data import PreConfig
from ocular.train import get_device

device = get_device()
cfg = PreConfig(384, 256, crop=True, curvature=True)

# The delivered checkpoint. See the top-level README for where to obtain it.
ckpt = config.ROOT / "experiments" / "convnext_final_e1.pt"
net = omodel.build_model("convnext_tiny", pretrained=False)
net.load_state_dict(torch.load(ckpt, map_location=device))
net = net.to(device).eval()

## The question

**Rationale.** A model can reach the right answer for the wrong reason. Because the three devices differ in appearance, a classifier can learn to read the device rather than the disease, which would not transfer to an unseen scanner. An activation map makes this visible. If the map sits on the retinal lesion the model is using the pathology, and if it sits on a border or a background texture it is using an artefact.

## Method

**Method.** Grad-CAM weights the activations of the last convolutional stage by the gradient of the predicted class, keeps the positive part and rescales it to a map in the range zero to one. The map is then placed over the scan. The layer and the computation live in `ocular.explain`, and the intended behaviour is documented in each function.

## One example per class

**Method.** One scan is drawn from each class of the clinic set and its map is shown beneath the plain scan. A model that has learned the pathology should place the map on the lesion for CNV, DME and DRUSEN, and spread it thinly for a normal scan.

In [ ]:
# TODO runs once ocular.explain.explain_scan and gradcam are implemented
frame = data._clinic_frame()
examples = {c: frame[frame["cls"] == c]["path"].iloc[0] for c in config.CLASSES}

fig, axes = plt.subplots(2, len(config.CLASSES), figsize=(12, 5))
for j, (cls, path) in enumerate(examples.items()):
    scan, cam = explain.explain_scan(net, path, cfg, device=device)
    axes[0, j].imshow(scan, cmap="gray")
    axes[0, j].set_title(cls, fontsize=10)
    axes[1, j].imshow(explain.overlay(scan, cam))
    for ax in (axes[0, j], axes[1, j]):
        ax.axis("off")
plt.tight_layout()
plt.show()

## Where the model looks on the drusen misses

**Rationale.** Notebook 3 found that the missed drusen scans are read as healthy with high confidence, which suggests the drusen signal is absent from the printed slice rather than merely under threshold. The activation map is a second view on that claim. If the map on a missed scan sits away from any drusen, or spreads with no focus, that supports a missing signal rather than a model that looked in the wrong place.

In [ ]:
# TODO runs once ocular.explain.explain_scan is implemented
clinic = pd.read_csv(config.ROOT / "experiments" / "results" / "clinic_scan_e1.csv")
missed = clinic[(clinic["truth"] == "DRUSEN") & (clinic["pred"] != "DRUSEN")].reset_index(drop=True)

n = len(missed)
fig, axes = plt.subplots(2, n, figsize=(2.2 * n, 5))
for j, row in missed.iterrows():
    path = data.CLINIC_DIR / row["name"]
    scan, cam = explain.explain_scan(net, path, cfg, device=device)
    axes[0, j].imshow(scan, cmap="gray")
    axes[0, j].set_title(f"{row['name']}\npredicted {row['pred']}", fontsize=7)
    axes[1, j].imshow(explain.overlay(scan, cam))
    for ax in (axes[0, j], axes[1, j]):
        ax.axis("off")
plt.tight_layout()
plt.show()

## Finding

**Finding.** To be written once the maps are produced. State whether the model attends to the pathology on the correct cases, and describe what the maps on the missed drusen show.